# fio DFS `rand_write` — Run Statistics

Reads all `fio_dfs_*.json` files in this directory and reports per-run values plus
summary statistics (N, mean, min, median, max) for:

- **Bandwidth** (GiB/s)
- **IOPS**
- **Mean latency** (ms, from `lat_ns.mean`)
- **CPU usage** — user + sys %

Configuration common to all runs: `ioengine=dfs`, `bs=1m`, `numjobs=16`, `iodepth=16`,
`rw=randwrite`, `runtime=60s`.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path(".")

In [ ]:
def parse_dfs_file(fpath):
    m = re.search(r"fio_dfs_bs(\S+?)_nj(\d+)_iod(\d+)_(\d+)\.json", Path(fpath).name)
    bs, nj, iod, ts = m.group(1), int(m.group(2)), int(m.group(3)), int(m.group(4))

    with open(fpath) as f:
        d = json.load(f)

    job = d["jobs"][0]          # group_reporting=1 → single aggregated job
    w   = job["write"]

    return dict(
        timestamp   = ts,
        block_size  = bs,
        numjobs     = nj,
        iodepth     = iod,
        bw_GiBs     = w["bw_bytes"] / 1024**3,
        iops        = w["iops"],
        lat_mean_ms = w["lat_ns"]["mean"] / 1e6,
        cpu         = job["usr_cpu"] + job["sys_cpu"],
    )


rows = [parse_dfs_file(p) for p in sorted(RESULTS_DIR.glob("fio_dfs_*.json"))]
df   = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
df.index.name = "run"
df.index += 1

print(f"Loaded {len(df)} DFS runs")
df

## Summary Statistics

In [ ]:
METRICS = {
    "bw_GiBs"    : "BW (GiB/s)",
    "iops"       : "IOPS",
    "lat_mean_ms": "Mean Latency (ms)",
    "cpu"        : "CPU (usr+sys) %",
}

stats = (
    df[list(METRICS)]
    .agg(["count", "mean", "min", "median", "max"])
    .rename(index={"count": "N"})
    .rename(columns=METRICS)
    .T
)
stats["N"] = stats["N"].astype(int)

pd.set_option("display.float_format", "{:.4f}".format)
stats

## Per-Run Overview

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.flatten()

plot_cols = [
    ("bw_GiBs",     "Bandwidth (GiB/s)"),
    ("iops",        "IOPS"),
    ("lat_mean_ms", "Mean Latency (ms)"),
    ("cpu",         "CPU (usr+sys) %"),
]

runs = df.index.tolist()

for ax, (col, label) in zip(axes, plot_cols):
    vals = df[col].values
    ax.bar(runs, vals, color="steelblue", alpha=0.8, edgecolor="white")
    ax.axhline(vals.mean(),   color="red",    linestyle="--", linewidth=1.2, label=f"mean={vals.mean():.3g}")
    ax.axhline(float(pd.Series(vals).median()), color="orange", linestyle=":",  linewidth=1.2, label=f"median={float(pd.Series(vals).median()):.3g}")
    ax.set_xlabel("Run #")
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.set_xticks(runs)
    ax.legend(fontsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.suptitle("fio DFS rand_write  |  bs=1m  nj=16  iod=16", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "fio_dfs_stats.pdf", dpi=150, bbox_inches="tight")
plt.show()